In [3]:
import cv2
import numpy as np
from pathlib import Path

# 폴더 경로 설정
base_dir = Path("C:/Data/computer_vision/YOLO3")

# 저장할 영상 파일명
video_path = str(base_dir / "yolo26.avi")

# YOLO 파일 경로
cfg_path = str(base_dir / "yolov3.cfg")
weights_path = str(base_dir / "yolov3.weights")
names_path = str(base_dir / "yolo.names")

# --------------------------------------------------
# 1. 노트북 카메라로 영상 촬영 및 저장
# --------------------------------------------------

capture = cv2.VideoCapture(0)

fourcc = cv2.VideoWriter_fourcc(*'XVID')
record = False
video = None

if not capture.isOpened():
    print("카메라를 열 수 없습니다.")
    exit()

print("카메라 촬영 화면입니다.")
print("1번 키: 녹화 시작")
print("2번 키: 녹화 중지")
print("ESC 키: 촬영 종료 후 YOLO 탐지 시작")

while True:
    ret, frame = capture.read()

    if not ret:
        print("카메라 프레임을 읽을 수 없습니다.")
        break

    cv2.imshow("CameraFrame", frame)

    key = cv2.waitKey(33)

    if key == 27:  # ESC 키
        print("촬영 종료")
        break

    elif key == 49:  # 1번 키
        if record == False:
            print("녹화 시작")
            record = True

            video = cv2.VideoWriter(
                video_path,
                fourcc,
                20.0,
                (frame.shape[1], frame.shape[0])
            )

    elif key == 50:  # 2번 키
        if record == True:
            print("녹화 중지")
            record = False

            if video is not None:
                video.release()
                video = None

    if record == True and video is not None:
        video.write(frame)

capture.release()

if video is not None:
    video.release()

cv2.destroyAllWindows()

print("영상 저장 완료:", video_path)

# --------------------------------------------------
# 2. 저장된 yolo26.avi 파일을 불러와서 YOLO 객체 탐지
# --------------------------------------------------

YOLO_net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)

classes = []

with open(names_path, 'r', encoding='utf-8') as f:
    classes = [line.strip() for line in f.readlines()]

layer_names = YOLO_net.getLayerNames()
output_layers = [layer_names[i - 1] for i in YOLO_net.getUnconnectedOutLayers().flatten()]

VideoSignal = cv2.VideoCapture(video_path)

if not VideoSignal.isOpened():
    print("저장된 yolo26.avi 파일을 열 수 없습니다.")
    exit()

print("YOLO 객체 탐지를 시작합니다.")
print("아무 키나 누르면 종료됩니다.")

while True:
    ret, frame = VideoSignal.read()

    if not ret:
        print("영상 재생이 끝났습니다.")
        break

    height, width, channel = frame.shape

    blob = cv2.dnn.blobFromImage(
        frame,
        0.00392,
        (416, 416),
        (0, 0, 0),
        True,
        crop=False
    )

    YOLO_net.setInput(blob)
    outs = YOLO_net.forward(output_layers)

    class_ids = []
    confidences = []
    boxes = []

    for out in outs:
        for detection in out:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]

            if confidence > 0.5:
                center_x = int(detection[0] * width)
                center_y = int(detection[1] * height)

                box_width = int(detection[2] * width)
                box_height = int(detection[3] * height)

                x = int(center_x - box_width / 2)
                y = int(center_y - box_height / 2)

                boxes.append([x, y, box_width, box_height])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.45, 0.4)

    if len(indexes) > 0:
        for i in indexes.flatten():
            x, y, box_width, box_height = boxes[i]

            label = str(classes[class_ids[i]])
            confidence = confidences[i]

            text = label + " " + str(round(confidence, 2))

            cv2.rectangle(
                frame,
                (x, y),
                (x + box_width, y + box_height),
                (0, 0, 255),
                2
            )

            cv2.putText(
                frame,
                text,
                (x, y - 10),
                cv2.FONT_ITALIC,
                0.6,
                (255, 255, 255),
                2
            )

    cv2.imshow("YOLOv3 Detection", frame)

    if cv2.waitKey(100) > 0:
        break

VideoSignal.release()
cv2.destroyAllWindows()

카메라 촬영 화면입니다.
1번 키: 녹화 시작
2번 키: 녹화 중지
ESC 키: 촬영 종료 후 YOLO 탐지 시작
녹화 시작
녹화 중지
촬영 종료
영상 저장 완료: C:\Data\computer_vision\YOLO3\yolo26.avi
YOLO 객체 탐지를 시작합니다.
아무 키나 누르면 종료됩니다.
